# Item-Item Collaborative Filtering (with Category Boost)

## Ý tưởng cốt lõi
Nếu **User 1** mua: `[iPhone case, AirPods, MagSafe]`  
Và **User 2** mua: `[iPhone case, AirPods]`  
→ Gợi ý User 2 mua **MagSafe** (vì rất nhiều user mua 2 thứ kia cũng mua MagSafe)

## Cải tiến: Category Boost
Khi predict, items cùng **category** với sản phẩm đã xem sẽ được
nhân thêm hệ số boost (mặc định 1.5x) → tránh gợi ý lạc category.

## Các bước
1. Đọc ratings theo **chunk** (dữ liệu lớn)
2. Lọc **10-core**
3. **Chuẩn hóa** rating theo user mean
4. Tính **cosine similarity S** theo batch
5. Lưu `item_categories` cùng S vào **pkl**

In [ ]:
import ast
import numpy as np
import pandas as pd
import pickle
from scipy import sparse
from sklearn.metrics.pairwise import cosine_similarity

## Bước 1 – Đọc dữ liệu theo chunk

In [ ]:
RATINGS_CSV = '../preprocessing/preprocessed_Cell_Phones_and_Accessories.csv'
META_CSV    = '../preprocessing/preprocessed_meta_preprocessedCell_Phones_and_Accessories.csv'
OUTPUT_PKL  = 'collaborative_model.pkl'

chunks = []
for chunk in pd.read_csv(RATINGS_CSV,
                         usecols=['user_id', 'parent_asin', 'rating'],
                         dtype={'rating': 'int8'},
                         chunksize=500_000):
    chunks.append(chunk)
    print(f'  Chunk: {len(chunk):,} dòng')

df = pd.concat(chunks, ignore_index=True)
print(f'Tổng cộng: {len(df):,} ratings')

## Bước 2 – Lọc 10-core

In [ ]:
for i in range(3):
    n = len(df)
    user_cnt = df['user_id'].value_counts()
    item_cnt = df['parent_asin'].value_counts()
    df = df[
        df['user_id'].isin(user_cnt[user_cnt >= 10].index) &
        df['parent_asin'].isin(item_cnt[item_cnt >= 10].index)
    ]
    print(f'Lần {i+1}: {n:,} → {len(df):,} ratings')

df = df.drop_duplicates(subset=['user_id', 'parent_asin'], keep='last')
print(f'Sau dedup: {len(df):,} ratings')

## Bước 3 – Index mapping + Metadata (title & categories)

In [ ]:
all_users = df['user_id'].unique()
all_items = df['parent_asin'].unique()

user_to_idx = {u: i for i, u in enumerate(all_users)}
item_to_idx = {it: i for i, it in enumerate(all_items)}
idx_to_item = {i: it for it, i in item_to_idx.items()}

n_users = len(all_users)
n_items = len(all_items)
print(f'Users: {n_users:,}  |  Items: {n_items:,}')

user_col = df['user_id'].map(user_to_idx).to_numpy(dtype=np.int32)
item_col = df['parent_asin'].map(item_to_idx).to_numpy(dtype=np.int32)
rate_col = df['rating'].to_numpy(dtype=np.float32)

# Đọc metadata: title + categories
df_meta = (pd.read_csv(META_CSV,
                        usecols=['parent_asin', 'title', 'categories'],
                        dtype=str)
             .drop_duplicates(subset=['parent_asin']))

title_map = dict(zip(df_meta['parent_asin'], df_meta['title'].fillna('')))
item_titles = [title_map.get(all_items[i], all_items[i]) for i in range(n_items)]

# Parse cột categories: chuỗi dạng "['A', 'B', 'C']" → list[str]
def parse_cats(s):
    try:
        v = ast.literal_eval(str(s))
        return [c.strip().lower() for c in v if c.strip()]
    except:
        return []

cat_map = {row['parent_asin']: parse_cats(row['categories'])
           for _, row in df_meta.iterrows()}

# item_categories[i] = list danh mục của item thứ i
item_categories = [cat_map.get(all_items[i], []) for i in range(n_items)]

print('Ví dụ item 0:')
print('  title     :', item_titles[0][:60])
print('  categories:', item_categories[0])

del df, df_meta

## Bước 4 – Chuẩn hóa rating theo user mean

In [ ]:
mu = np.zeros(n_users, dtype=np.float32)
for u in range(n_users):
    mask = (user_col == u)
    if mask.any():
        mu[u] = rate_col[mask].mean()

rate_norm = rate_col - mu[user_col]

Ybar = sparse.csr_matrix(
    (rate_norm, (item_col, user_col)),
    shape=(n_items, n_users),
    dtype=np.float32
)
print(f'Ybar shape: {Ybar.shape}  |  non-zeros: {Ybar.nnz:,}')

## Bước 5 – Tính ma trận Similarity S

In [ ]:
BATCH = 500
S = np.zeros((n_items, n_items), dtype=np.float32)

for start in range(0, n_items, BATCH):
    end = min(start + BATCH, n_items)
    S[start:end] = cosine_similarity(Ybar[start:end], Ybar).astype(np.float32)
    print(f'  {end:>6,} / {n_items:,} ({100*end/n_items:.0f}%)', end='\r')

print(f'\nS shape: {S.shape}')

## Bước 6 – Lưu pkl

In [ ]:
model_data = {
    'user_to_idx':  user_to_idx,
    'item_to_idx':  item_to_idx,
    'idx_to_item':  idx_to_item,
    'item_ids':     all_items,
    'item_titles':  item_titles,
    'item_categories': item_categories,  # Cột danh mục mới thêm
    'cf_S':    S,
    'cf_mu':   mu,
    'cf_k':    10,
    'Y_data':  np.column_stack([user_col, item_col, rate_col]),
}

with open(OUTPUT_PKL, 'wb') as f:
    pickle.dump(model_data, f, protocol=4)

import os
mb = os.path.getsize(OUTPUT_PKL) / 1024**2
print(f'✅ Đã lưu: {OUTPUT_PKL} ({mb:.0f} MB)')

## Kiểm tra nhanh – Thử recommend với Category Boost

In [ ]:
def recommend_with_category(viewed_item_indices, top_k=10, boost=1.5):
    agg = np.zeros(n_items, dtype=np.float32)
    viewed_cats = set()
    
    # Lấy tập hợp các category của sản phẩm đã xem
    for i in viewed_item_indices:
        for c in item_categories[i]:
            viewed_cats.add(c)

    for i in viewed_item_indices:
        row = S[i].copy()
        topk_idx = np.argpartition(row, -10)[-10:]
        mask = np.zeros_like(row)
        mask[topk_idx] = row[topk_idx]
        agg += mask

    # Áp dụng boost
    if viewed_cats:
        for i in range(n_items):
            if agg[i] > 0:
                # Nếu item i có category trùng với category đã xem
                if any(c in viewed_cats for c in item_categories[i]):
                    agg[i] *= boost

    agg[viewed_item_indices] = -np.inf
    ranked = np.argsort(agg)[::-1][:top_k]
    return [(item_titles[i], agg[i]) for i in ranked]

sample_viewed = [0, 1, 2]
print('Đã xem:')
for i in sample_viewed:
    print(f'  - {item_titles[i][:70]} (Cats: {item_categories[i][:2]})')

print('\nGợi ý (Boost 1.5x):')
for i, (t, score) in enumerate(recommend_with_category(sample_viewed), 1):
    print(f'  {i:2}. {t[:70]} | Điểm: {score:.2f}')